# Notebook-first application walkthrough

**Problem / objective:** Convert governed commerce data into an executive BI layer with auditable KPI definitions and reusable semantic-model assets.

**Decision / solution:** Surface the few KPIs and segments that require executive action, then retain drill-down evidence for investigation.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'executive_commerce_bi'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Surface the few KPIs and segments that require executive action, then retain drill-down evidence for investigation.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# Executive Commerce Intelligence — Full Python Code

**Hiring purpose:** one project, one notebook, with the actual Python implementation visible. The modular files remain in the repository because that is how production code should be organised; this notebook mirrors those files so a recruiter can inspect the full code without hunting.


## Dataset and reproducibility

Governed Olist warehouse exports used by the Power BI and Tableau layer.

The project README/data card documents provenance, constraints and the exact reproduction path. Large third-party raw files are not duplicated in Git when licensing or repository size makes that poor engineering practice.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_SLUG = 'executive_commerce_bi'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    target = Path('/content/uni_projects')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Jorgoluka100/uni_projects.git', str(target)], check=True)
    os.chdir(target)
    ROOT = target
PROJECT = ROOT / 'projects' / PROJECT_SLUG
assert PROJECT.exists(), PROJECT
print('Project:', PROJECT.resolve())


## Full Python implementation

Every code cell below is copied directly from the corresponding `.py` file on the same commit. These cells are intentionally tagged `source-mirror` so the notebook acts as a readable code portfolio while the canonical modules remain testable files.


### `prepare_bi_data.py`


In [ ]:
"""Prepare governed Power BI / Tableau exports from verified e-commerce Parquet marts."""
from __future__ import annotations

import argparse
import hashlib
import json
import tempfile
from pathlib import Path

import pandas as pd

REQUIRED_COLUMNS = {
    "headline_kpis": {
        "commercial_orders", "unique_customers", "merchandise_value_brl",
        "average_order_value_brl", "freight_value_brl", "total_payment_value_brl",
    },
    "customer_order_frequency": {
        "customers", "repeat_customers", "repeat_customer_pct",
        "average_orders_per_customer", "average_customer_merchandise_value_brl",
    },
    "seller_concentration_summary": {
        "active_sellers", "largest_seller_share_pct", "top_ten_seller_share_pct", "hhi_index",
    },
    "monthly_performance": {
        "order_month", "orders", "customers", "merchandise_value_brl",
        "average_order_value_brl", "merchandise_value_mom_pct", "merchandise_value_rank",
    },
    "category_performance": {
        "category_name", "orders", "items", "merchandise_value_brl", "freight_value_brl",
        "freight_to_merchandise_pct", "average_review_score", "merchandise_value_rank",
    },
    "delivery_review_summary": {
        "delivery_status", "delivered_orders", "average_review_score",
        "average_days_late", "delivered_order_share_pct",
    },
    "payment_behaviour": {
        "payment_type", "payment_rows", "orders", "payment_value_brl",
        "average_installments", "order_penetration_pct",
    },
    "top_categories_by_customer_state": {
        "customer_state", "category_name", "orders", "merchandise_value_brl", "category_rank",
    },
}


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read_verified(source_dir: Path, name: str) -> pd.DataFrame:
    path = source_dir / f"{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Missing verified upstream table: {path}")
    frame = pd.read_parquet(path)
    missing = REQUIRED_COLUMNS[name] - set(frame.columns)
    if missing:
        raise ValueError(f"{name} missing required columns: {sorted(missing)}")
    return frame


def build_outputs(source_dir: Path, output_dir: Path) -> dict[str, object]:
    output_dir.mkdir(parents=True, exist_ok=True)
    tables = {name: read_verified(source_dir, name) for name in REQUIRED_COLUMNS}

    headline = tables["headline_kpis"].iloc[0].to_dict()
    repeat = tables["customer_order_frequency"].iloc[0].to_dict()
    concentration = tables["seller_concentration_summary"].iloc[0].to_dict()
    executive = pd.DataFrame([{**headline, **repeat, **concentration}])

    export_frames = {
        "executive_kpis": executive,
        "monthly_performance": tables["monthly_performance"].sort_values("order_month"),
        "category_performance": tables["category_performance"].sort_values("merchandise_value_rank").head(20),
        "delivery_review_summary": tables["delivery_review_summary"],
        "payment_behaviour": tables["payment_behaviour"],
        "state_category_mix": tables["top_categories_by_customer_state"],
    }

    long_rows: list[dict[str, object]] = []
    for label, column in [
        ("Commercial Orders", "commercial_orders"),
        ("Unique Customers", "unique_customers"),
        ("Merchandise Value (BRL)", "merchandise_value_brl"),
        ("Average Order Value (BRL)", "average_order_value_brl"),
        ("Repeat Customer %", "repeat_customer_pct"),
        ("Top 10 Seller Share %", "top_ten_seller_share_pct"),
    ]:
        long_rows.append({"section": "Executive KPI", "dimension": label, "metric": label, "value": executive.iloc[0][column], "secondary_value": None, "sort_order": len(long_rows) + 1})

    for _, row in tables["monthly_performance"].iterrows():
        month = pd.Timestamp(row["order_month"]).strftime("%Y-%m")
        for metric, column in [("Monthly GMV", "merchandise_value_brl"), ("Monthly Orders", "orders"), ("Average Order Value", "average_order_value_brl")]:
            long_rows.append({"section": "Trend", "dimension": month, "metric": metric, "value": row[column], "secondary_value": row.get("merchandise_value_mom_pct"), "sort_order": int(pd.Timestamp(row["order_month"]).strftime("%Y%m"))})

    for _, row in tables["category_performance"].sort_values("merchandise_value_rank").head(15).iterrows():
        for metric, column in [("Category GMV", "merchandise_value_brl"), ("Category Orders", "orders"), ("Average Review", "average_review_score")]:
            long_rows.append({"section": "Category", "dimension": row["category_name"], "metric": metric, "value": row[column], "secondary_value": row["freight_to_merchandise_pct"], "sort_order": int(row["merchandise_value_rank"])})

    for _, row in tables["delivery_review_summary"].iterrows():
        long_rows.append({"section": "Delivery", "dimension": row["delivery_status"], "metric": "Average Review", "value": row["average_review_score"], "secondary_value": row["delivered_order_share_pct"], "sort_order": 1 if row["delivery_status"] == "on_time" else 2})

    long_frame = pd.DataFrame(long_rows)
    export_frames["tableau_dashboard_long"] = long_frame

    manifest: dict[str, object] = {"source": "verified ecommerce_sql_analytics Parquet exports", "files": {}}
    for name, frame in export_frames.items():
        destination = output_dir / f"{name}.csv"
        frame.to_csv(destination, index=False, date_format="%Y-%m-%d")
        manifest["files"][destination.name] = {
            "rows": int(len(frame)),
            "columns": list(frame.columns),
            "sha256": sha256(destination),
        }

    manifest_path = output_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    return manifest


def self_test() -> None:
    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)
        source = root / "source"
        output = root / "output"
        source.mkdir()
        fixtures = {
            "headline_kpis": pd.DataFrame([{"commercial_orders": 3, "unique_customers": 2, "merchandise_value_brl": 250.0, "average_order_value_brl": 83.33, "freight_value_brl": 25.0, "total_payment_value_brl": 275.0}]),
            "customer_order_frequency": pd.DataFrame([{"customers": 2, "repeat_customers": 1, "repeat_customer_pct": 50.0, "average_orders_per_customer": 1.5, "average_customer_merchandise_value_brl": 125.0}]),
            "seller_concentration_summary": pd.DataFrame([{"active_sellers": 2, "largest_seller_share_pct": 70.0, "top_ten_seller_share_pct": 100.0, "hhi_index": 5800.0}]),
            "monthly_performance": pd.DataFrame([{"order_month": pd.Timestamp("2017-01-01"), "orders": 3, "customers": 2, "merchandise_value_brl": 250.0, "average_order_value_brl": 83.33, "merchandise_value_mom_pct": None, "merchandise_value_rank": 1}]),
            "category_performance": pd.DataFrame([{"category_name": "health_beauty", "orders": 3, "items": 4, "merchandise_value_brl": 250.0, "freight_value_brl": 25.0, "freight_to_merchandise_pct": 10.0, "average_review_score": 4.2, "merchandise_value_rank": 1}]),
            "delivery_review_summary": pd.DataFrame([{"delivery_status": "on_time", "delivered_orders": 2, "average_review_score": 4.5, "average_days_late": 0.0, "delivered_order_share_pct": 66.67}, {"delivery_status": "late", "delivered_orders": 1, "average_review_score": 2.0, "average_days_late": 5.0, "delivered_order_share_pct": 33.33}]),
            "payment_behaviour": pd.DataFrame([{"payment_type": "credit_card", "payment_rows": 3, "orders": 3, "payment_value_brl": 275.0, "average_installments": 2.0, "order_penetration_pct": 100.0}]),
            "top_categories_by_customer_state": pd.DataFrame([{"customer_state": "SP", "category_name": "health_beauty", "orders": 3, "merchandise_value_brl": 250.0, "category_rank": 1}]),
        }
        for name, frame in fixtures.items():
            frame.to_parquet(source / f"{name}.parquet", index=False)
        manifest = build_outputs(source, output)
        assert (output / "executive_kpis.csv").exists()
        assert (output / "tableau_dashboard_long.csv").exists()
        assert manifest["files"]["executive_kpis.csv"]["rows"] == 1
        print("Executive Commerce BI data-contract self-test passed.")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Prepare governed exports for Power BI and Tableau")
    parser.add_argument("--source-dir", type=Path, default=Path("../ecommerce_sql_analytics/artifacts/tables"))
    parser.add_argument("--output-dir", type=Path, default=Path("data"))
    parser.add_argument("--self-test", action="store_true")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if args.self_test:
        self_test()
    else:
        manifest = build_outputs(args.source_dir, args.output_dir)
        print(json.dumps(manifest, indent=2))


if __name__ == "__main__":
    main()


### `business_analysis.py`


In [ ]:
"""Decision-oriented business analysis for the Executive Commerce BI project.

The dashboard should answer business questions, not just display charts. This module
runs on the verified DuckDB warehouse and produces compact, inspectable outputs for
Power BI, Tableau and the GitHub project page.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any

import pandas as pd


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _write_csv(frame: pd.DataFrame, output_dir: Path, filename: str, manifest: dict[str, Any]) -> None:
    destination = output_dir / filename
    frame.to_csv(destination, index=False, date_format="%Y-%m-%d")
    manifest.setdefault("files", {})[filename] = {
        "rows": int(len(frame)),
        "columns": list(frame.columns),
        "sha256": sha256(destination),
    }


def build_business_analysis(con, output_dir: Path, manifest: dict[str, Any]) -> dict[str, Any]:
    """Create the analysis tables and a recruiter-readable insight summary."""
    output_dir.mkdir(parents=True, exist_ok=True)

    monthly = con.execute(
        """
        SELECT
            order_month,
            orders,
            customers,
            merchandise_value_brl,
            average_order_value_brl,
            merchandise_value_mom_pct,
            merchandise_value_rank
        FROM analytics.monthly_performance
        ORDER BY order_month
        """
    ).df()

    customer_segments = con.execute(
        """
        SELECT
            customer_segment,
            COUNT(*)::BIGINT AS customers,
            SUM(orders)::BIGINT AS orders,
            ROUND(SUM(merchandise_value_brl), 2) AS merchandise_value_brl,
            ROUND(AVG(orders), 2) AS average_orders_per_customer,
            ROUND(AVG(merchandise_value_brl), 2) AS average_customer_value_brl,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS customer_share_pct,
            ROUND(100.0 * SUM(merchandise_value_brl) / SUM(SUM(merchandise_value_brl)) OVER (), 2) AS merchandise_value_share_pct
        FROM analytics.customer_value_segments
        GROUP BY customer_segment
        ORDER BY merchandise_value_brl DESC
        """
    ).df()

    delivery_buckets = con.execute(
        """
        WITH scoped AS (
            SELECT
                order_id,
                merchandise_value_brl,
                review_score,
                days_late,
                CASE
                    WHEN delivered_late = FALSE THEN 'On time'
                    WHEN days_late BETWEEN 1 AND 3 THEN '1-3 days late'
                    WHEN days_late BETWEEN 4 AND 7 THEN '4-7 days late'
                    WHEN days_late BETWEEN 8 AND 14 THEN '8-14 days late'
                    ELSE '15+ days late'
                END AS delivery_bucket,
                CASE
                    WHEN delivered_late = FALSE THEN 1
                    WHEN days_late BETWEEN 1 AND 3 THEN 2
                    WHEN days_late BETWEEN 4 AND 7 THEN 3
                    WHEN days_late BETWEEN 8 AND 14 THEN 4
                    ELSE 5
                END AS bucket_order
            FROM analytics.order_mart
            WHERE commercial_order
              AND delivered_late IS NOT NULL
        )
        SELECT
            delivery_bucket,
            COUNT(*)::BIGINT AS orders,
            ROUND(SUM(merchandise_value_brl), 2) AS merchandise_value_brl,
            ROUND(AVG(days_late), 2) AS average_days_late,
            ROUND(AVG(review_score), 2) AS average_review_score,
            ROUND(100.0 * AVG(CASE WHEN review_score = 1 THEN 1.0 ELSE 0.0 END), 2) AS one_star_review_pct,
            ROUND(100.0 * AVG(CASE WHEN review_score = 5 THEN 1.0 ELSE 0.0 END), 2) AS five_star_review_pct,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS delivered_order_share_pct
        FROM scoped
        GROUP BY delivery_bucket, bucket_order
        ORDER BY bucket_order
        """
    ).df()

    state_analysis = con.execute(
        """
        SELECT
            customer_state,
            COUNT(*)::BIGINT AS orders,
            COUNT(DISTINCT customer_unique_id)::BIGINT AS unique_customers,
            ROUND(SUM(merchandise_value_brl), 2) AS merchandise_value_brl,
            ROUND(AVG(merchandise_value_brl), 2) AS average_order_value_brl,
            ROUND(100.0 * AVG(CASE WHEN delivered_late THEN 1.0 WHEN delivered_late = FALSE THEN 0.0 END), 2) AS late_delivery_pct,
            ROUND(SUM(CASE WHEN delivered_late THEN merchandise_value_brl ELSE 0 END), 2) AS late_order_merchandise_value_brl,
            ROUND(AVG(review_score), 2) AS average_review_score,
            ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1.0 ELSE 0.0 END), 2) AS low_review_order_pct
        FROM analytics.order_mart
        WHERE commercial_order
          AND customer_state IS NOT NULL
        GROUP BY customer_state
        ORDER BY merchandise_value_brl DESC
        """
    ).df()

    category_analysis = con.execute(
        """
        WITH order_category AS (
            SELECT
                order_id,
                category_name,
                SUM(item_price_brl) AS merchandise_value_brl,
                SUM(item_freight_brl) AS freight_value_brl,
                MAX(CASE WHEN delivered_late THEN 1 WHEN delivered_late = FALSE THEN 0 END) AS delivered_late,
                MAX(review_score) AS review_score
            FROM analytics.item_mart
            WHERE commercial_order
            GROUP BY order_id, category_name
        )
        SELECT
            category_name,
            COUNT(*)::BIGINT AS orders,
            ROUND(SUM(merchandise_value_brl), 2) AS merchandise_value_brl,
            ROUND(SUM(freight_value_brl), 2) AS freight_value_brl,
            ROUND(100.0 * SUM(freight_value_brl) / NULLIF(SUM(merchandise_value_brl), 0), 2) AS freight_to_merchandise_pct,
            ROUND(100.0 * AVG(delivered_late), 2) AS late_delivery_pct,
            ROUND(SUM(CASE WHEN delivered_late = 1 THEN merchandise_value_brl ELSE 0 END), 2) AS late_order_merchandise_value_brl,
            ROUND(AVG(review_score), 2) AS average_review_score,
            ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1.0 ELSE 0.0 END), 2) AS low_review_order_pct
        FROM order_category
        GROUP BY category_name
        HAVING COUNT(*) >= 100
        ORDER BY merchandise_value_brl DESC
        """
    ).df()

    seller_risk = con.execute(
        """
        SELECT
            operational_status,
            COUNT(*)::BIGINT AS sellers,
            SUM(orders)::BIGINT AS seller_orders,
            ROUND(SUM(merchandise_value_brl), 2) AS merchandise_value_brl,
            ROUND(AVG(late_delivery_rate_pct), 2) AS average_late_delivery_rate_pct,
            ROUND(AVG(average_review_score), 2) AS average_review_score
        FROM analytics.seller_operational_review
        GROUP BY operational_status
        ORDER BY CASE WHEN operational_status = 'review_priority' THEN 0 ELSE 1 END
        """
    ).df()

    payment = con.execute(
        "SELECT * FROM analytics.payment_behaviour ORDER BY payment_value_brl DESC"
    ).df()

    # Transparent management priority: rank material value currently exposed to late delivery.
    state_priority = state_analysis.loc[state_analysis["orders"] >= 500].copy()
    state_priority["entity_type"] = "state"
    state_priority["entity_name"] = state_priority["customer_state"]
    state_priority = state_priority.sort_values("late_order_merchandise_value_brl", ascending=False)

    category_priority = category_analysis.loc[category_analysis["orders"] >= 500].copy()
    category_priority["entity_type"] = "category"
    category_priority["entity_name"] = category_priority["category_name"]
    category_priority = category_priority.sort_values("late_order_merchandise_value_brl", ascending=False)

    priority_columns = [
        "entity_type",
        "entity_name",
        "orders",
        "merchandise_value_brl",
        "late_delivery_pct",
        "late_order_merchandise_value_brl",
        "average_review_score",
        "low_review_order_pct",
    ]
    operational_priority = pd.concat(
        [state_priority[priority_columns], category_priority[priority_columns]],
        ignore_index=True,
    ).sort_values("late_order_merchandise_value_brl", ascending=False)
    operational_priority.insert(0, "priority_rank", range(1, len(operational_priority) + 1))

    _write_csv(monthly, output_dir, "analysis_monthly_growth.csv", manifest)
    _write_csv(customer_segments, output_dir, "analysis_customer_segments.csv", manifest)
    _write_csv(delivery_buckets, output_dir, "analysis_delivery_impact.csv", manifest)
    _write_csv(state_analysis, output_dir, "analysis_state_performance.csv", manifest)
    _write_csv(category_analysis, output_dir, "analysis_category_performance.csv", manifest)
    _write_csv(seller_risk, output_dir, "analysis_seller_risk.csv", manifest)
    _write_csv(payment, output_dir, "analysis_payment_mix.csv", manifest)
    _write_csv(operational_priority, output_dir, "analysis_operational_priority.csv", manifest)

    headline = con.execute("SELECT * FROM analytics.headline_kpis").df().iloc[0]
    repeat = con.execute("SELECT * FROM analytics.customer_order_frequency").df().iloc[0]
    delivery = con.execute("SELECT * FROM analytics.delivery_review_summary").df()
    on_time = delivery.loc[delivery["delivery_status"] == "on_time"].iloc[0]
    late = delivery.loc[delivery["delivery_status"] == "late"].iloc[0]

    strongest_month = monthly.sort_values("merchandise_value_brl", ascending=False).iloc[0]
    top_state = state_analysis.iloc[0]
    top_category = category_analysis.iloc[0]
    priority = operational_priority.iloc[0] if not operational_priority.empty else None
    repeat_segments = customer_segments[customer_segments["customer_segment"].str.startswith("repeat")]

    summary: dict[str, Any] = {
        "scope": {
            "commercial_orders": int(headline["commercial_orders"]),
            "unique_customers": int(headline["unique_customers"]),
            "merchandise_value_brl": round(float(headline["merchandise_value_brl"]), 2),
        },
        "growth": {
            "strongest_complete_month": pd.Timestamp(strongest_month["order_month"]).strftime("%Y-%m"),
            "strongest_month_merchandise_value_brl": round(float(strongest_month["merchandise_value_brl"]), 2),
            "strongest_month_orders": int(strongest_month["orders"]),
        },
        "customers": {
            "repeat_customer_pct": round(float(repeat["repeat_customer_pct"]), 2),
            "repeat_segment_merchandise_value_share_pct": round(float(repeat_segments["merchandise_value_share_pct"].sum()), 2),
        },
        "delivery": {
            "late_delivery_share_pct": round(float(late["delivered_order_share_pct"]), 2),
            "on_time_average_review": round(float(on_time["average_review_score"]), 2),
            "late_average_review": round(float(late["average_review_score"]), 2),
            "review_score_gap": round(float(on_time["average_review_score"] - late["average_review_score"]), 2),
        },
        "market": {
            "top_state": str(top_state["customer_state"]),
            "top_state_merchandise_value_brl": round(float(top_state["merchandise_value_brl"]), 2),
            "top_category": str(top_category["category_name"]),
            "top_category_merchandise_value_brl": round(float(top_category["merchandise_value_brl"]), 2),
        },
        "management_priority": None if priority is None else {
            "entity_type": str(priority["entity_type"]),
            "entity_name": str(priority["entity_name"]),
            "late_order_merchandise_value_brl": round(float(priority["late_order_merchandise_value_brl"]), 2),
            "late_delivery_pct": round(float(priority["late_delivery_pct"]), 2),
            "average_review_score": round(float(priority["average_review_score"]), 2),
        },
    }

    summary_path = output_dir / "analysis_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    manifest.setdefault("files", {})[summary_path.name] = {
        "rows": 1,
        "columns": list(summary.keys()),
        "sha256": sha256(summary_path),
    }

    priority_text = "No material late-delivery priority was produced."
    if priority is not None:
        priority_text = (
            f"The largest transparent operational exposure is **{priority['entity_type']} "
            f"{priority['entity_name']}**, with **R${float(priority['late_order_merchandise_value_brl']):,.0f}** "
            f"of merchandise value attached to late orders, a **{float(priority['late_delivery_pct']):.1f}%** "
            f"late-delivery rate and **{float(priority['average_review_score']):.2f}/5** average review score."
        )

    insights = f"""# Verified Business Analysis\n\nThis analysis is generated from the pinned Olist warehouse used by the Power BI and Tableau project. It is designed around management questions rather than chart count.\n\n## Executive readout\n\n- **Commercial scope:** {int(headline['commercial_orders']):,} orders, {int(headline['unique_customers']):,} unique customers and **R${float(headline['merchandise_value_brl']):,.2f}** merchandise value.\n- **Peak complete month:** {pd.Timestamp(strongest_month['order_month']).strftime('%Y-%m')} with **R${float(strongest_month['merchandise_value_brl']):,.2f}** GMV across {int(strongest_month['orders']):,} orders.\n- **Retention challenge:** only **{float(repeat['repeat_customer_pct']):.2f}%** of customers repeat, while repeat-customer segments account for **{float(repeat_segments['merchandise_value_share_pct'].sum()):.2f}%** of merchandise value.\n- **Delivery matters:** late deliveries represent **{float(late['delivered_order_share_pct']):.2f}%** of delivered commercial orders. Average review score falls from **{float(on_time['average_review_score']):.2f}/5** on time to **{float(late['average_review_score']):.2f}/5** when late — a **{float(on_time['average_review_score'] - late['average_review_score']):.2f}-point gap**.\n- **Largest market:** {top_state['customer_state']} contributes **R${float(top_state['merchandise_value_brl']):,.2f}** GMV.\n- **Largest analysed category:** {top_category['category_name']} contributes **R${float(top_category['merchandise_value_brl']):,.2f}** GMV.\n\n## Management priority\n\n{priority_text}\n\nThe priority table deliberately ranks **late-order merchandise value**, not an opaque composite score. A manager can therefore see exactly why an entity is high priority and then use late-delivery rate, review score and low-review share as context.\n\n## Dashboard questions\n\n1. Is commercial value growing, and which months drive the change?\n2. How much value comes from repeat customers versus one-time customers?\n3. How sharply does customer satisfaction deteriorate as delivery delay increases?\n4. Which states and categories combine material value with operational risk?\n5. Which sellers require operational review?\n6. How do customers pay, and where is instalment behaviour concentrated?\n\n## Generated evidence\n\nThe `data/analysis_*.csv` files and `analysis_summary.json` are regenerated by GitHub Actions from the pinned source dataset and checked alongside the Power BI/Tableau source contracts.\n"""
    insights_path = output_dir.parent / "VERIFIED_ANALYSIS.md"
    insights_path.write_text(insights, encoding="utf-8")

    return summary


### `cohort_analysis.py`


In [ ]:
"""Generate correctly right-censored cohort-retention evidence.

A sparse retention table can omit both future months and observed months with zero
returning customers. That is useful for storage but unsafe for an aggregate curve:
zero-retention observed cohorts would disappear from the denominator. This module
builds the complete *eligible* cohort-age grid, fills observed zero-return months
with zero, and excludes only months that were not yet observable.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import duckdb
import pandas as pd

PROJECT = Path(__file__).resolve().parent
DATA = PROJECT / "data"
DB_PATH = PROJECT / ".artifacts" / "ecommerce.duckdb"
MANIFEST = DATA / "manifest.json"
SUMMARY = DATA / "analysis_summary.json"
VERIFIED_ANALYSIS = PROJECT / "VERIFIED_ANALYSIS.md"
COMPLETE_MONTH_EXCLUSIVE = "2018-09-01"


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def register_file(manifest: dict, path: Path, frame: pd.DataFrame) -> None:
    manifest.setdefault("files", {})[path.name] = {
        "rows": int(len(frame)),
        "columns": list(frame.columns),
        "sha256": sha256(path),
    }


def build_right_censored_cohorts(con: duckdb.DuckDBPyConnection) -> pd.DataFrame:
    return con.execute(
        f"""
        WITH customer_months AS (
            SELECT DISTINCT
                customer_unique_id,
                order_month
            FROM analytics.order_mart
            WHERE commercial_order
              AND customer_unique_id IS NOT NULL
              AND order_month < DATE '{COMPLETE_MONTH_EXCLUSIVE}'
        ),
        customer_cohorts AS (
            SELECT
                customer_unique_id,
                MIN(order_month) AS cohort_month
            FROM customer_months
            GROUP BY customer_unique_id
        ),
        cohort_sizes AS (
            SELECT
                cohort_month,
                COUNT(*)::BIGINT AS cohort_customers
            FROM customer_cohorts
            WHERE cohort_month >= DATE '2017-01-01'
              AND cohort_month < DATE '{COMPLETE_MONTH_EXCLUSIVE}'
            GROUP BY cohort_month
        ),
        activity AS (
            SELECT
                c.cohort_month,
                DATE_DIFF('month', c.cohort_month, m.order_month)::INTEGER AS month_number,
                COUNT(DISTINCT m.customer_unique_id)::BIGINT AS active_customers
            FROM customer_months m
            JOIN customer_cohorts c USING (customer_unique_id)
            WHERE c.cohort_month >= DATE '2017-01-01'
              AND c.cohort_month < DATE '{COMPLETE_MONTH_EXCLUSIVE}'
              AND DATE_DIFF('month', c.cohort_month, m.order_month) BETWEEN 0 AND 12
            GROUP BY c.cohort_month, month_number
        ),
        ages AS (
            SELECT range::INTEGER AS month_number
            FROM range(0, 13)
        ),
        eligible_grid AS (
            SELECT
                s.cohort_month,
                a.month_number,
                s.cohort_customers
            FROM cohort_sizes s
            CROSS JOIN ages a
            WHERE s.cohort_month + a.month_number * INTERVAL '1 month'
                  < DATE '{COMPLETE_MONTH_EXCLUSIVE}'
        )
        SELECT
            g.cohort_month,
            g.month_number,
            COALESCE(a.active_customers, 0)::BIGINT AS active_customers,
            g.cohort_customers,
            ROUND(100.0 * COALESCE(a.active_customers, 0) / NULLIF(g.cohort_customers, 0), 2) AS retention_pct
        FROM eligible_grid g
        LEFT JOIN activity a
          ON a.cohort_month = g.cohort_month
         AND a.month_number = g.month_number
        ORDER BY g.cohort_month, g.month_number
        """
    ).df()


def build_weighted_curve(raw: pd.DataFrame) -> pd.DataFrame:
    grouped = raw.groupby("month_number", as_index=False).agg(
        observable_cohorts=("cohort_month", "count"),
        active_customers=("active_customers", "sum"),
        eligible_cohort_customers=("cohort_customers", "sum"),
        average_cohort_retention_pct=("retention_pct", "mean"),
    )
    grouped["weighted_retention_pct"] = (
        100.0 * grouped["active_customers"] / grouped["eligible_cohort_customers"]
    ).round(2)
    grouped["average_cohort_retention_pct"] = grouped["average_cohort_retention_pct"].round(2)
    return grouped[
        [
            "month_number",
            "observable_cohorts",
            "active_customers",
            "eligible_cohort_customers",
            "weighted_retention_pct",
            "average_cohort_retention_pct",
        ]
    ].sort_values("month_number")


def main() -> None:
    if not DB_PATH.exists():
        raise FileNotFoundError(f"Verified warehouse not found: {DB_PATH}")

    con = duckdb.connect(str(DB_PATH), read_only=True)
    raw = build_right_censored_cohorts(con)
    con.close()
    curve = build_weighted_curve(raw)

    if raw.empty or curve.empty:
        raise AssertionError("Cohort retention analysis produced no rows")
    month_zero = curve.loc[curve["month_number"] == 0].iloc[0]
    if float(month_zero["weighted_retention_pct"]) != 100.0:
        raise AssertionError("Month-zero weighted retention must equal 100%")
    if not curve["observable_cohorts"].is_monotonic_decreasing:
        raise AssertionError("Eligible cohort count should not increase with cohort age")
    if (raw["active_customers"] > raw["cohort_customers"]).any():
        raise AssertionError("Active customers cannot exceed cohort size")
    if not (raw["active_customers"] == 0).any():
        raise AssertionError("Expected observed zero-retention cohort-months in complete grid")

    raw_path = DATA / "analysis_cohort_retention.csv"
    curve_path = DATA / "analysis_retention_curve.csv"
    raw.to_csv(raw_path, index=False, date_format="%Y-%m-%d")
    curve.to_csv(curve_path, index=False)

    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    register_file(manifest, raw_path, raw)
    register_file(manifest, curve_path, curve)

    summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
    retention_summary: dict[str, float | int] = {}
    for month in [1, 3, 6, 12]:
        match = curve.loc[curve["month_number"] == month]
        if not match.empty:
            row = match.iloc[0]
            retention_summary[f"month_{month}_weighted_retention_pct"] = round(float(row["weighted_retention_pct"]), 2)
            retention_summary[f"month_{month}_observable_cohorts"] = int(row["observable_cohorts"])
    summary["cohort_retention"] = retention_summary
    SUMMARY.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    manifest["business_analysis"] = summary
    manifest["files"][SUMMARY.name]["sha256"] = sha256(SUMMARY)
    MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    m1 = retention_summary["month_1_weighted_retention_pct"]
    m3 = retention_summary["month_3_weighted_retention_pct"]
    m6 = retention_summary["month_6_weighted_retention_pct"]
    section = (
        "\n## Cohort retention\n\n"
        "The retention curve uses an **eligible cohort-age grid**: observed months with no returning customers are recorded as zero, while genuinely future months are excluded. This avoids both zero-fill censoring bias and sparse-table denominator bias.\n\n"
        f"- Weighted month-1 retention: **{m1:.2f}%**\n"
        f"- Weighted month-3 retention: **{m3:.2f}%**\n"
        f"- Weighted month-6 retention: **{m6:.2f}%**\n\n"
        "The low cohort retention reinforces the repeat-customer finding without making recent cohorts look worse simply because their future follow-up months do not yet exist.\n"
    )
    analysis_text = VERIFIED_ANALYSIS.read_text(encoding="utf-8")
    marker = "\n## Cohort retention\n"
    if marker in analysis_text:
        analysis_text = analysis_text.split(marker, 1)[0].rstrip() + "\n"
    VERIFIED_ANALYSIS.write_text(analysis_text.rstrip() + "\n" + section, encoding="utf-8")

    print(json.dumps(retention_summary, indent=2))


if __name__ == "__main__":
    main()


### `enrich_tableau_analysis.py`


In [ ]:
"""Add verified customer/service analysis to Tableau's tidy long-form extract."""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd

PROJECT = Path(__file__).resolve().parent
DATA = PROJECT / "data"
TABLEAU = DATA / "tableau_dashboard_long.csv"
MANIFEST = DATA / "manifest.json"
DEEP_SECTIONS = {"Customer Segment", "Cohort Retention", "Delay Impact", "Operational Priority"}


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main() -> None:
    base = pd.read_csv(TABLEAU)
    base = base.loc[~base["section"].isin(DEEP_SECTIONS)].copy()
    next_sort = int(base["sort_order"].max()) + 1
    rows: list[dict[str, object]] = []

    customer = pd.read_csv(DATA / "analysis_customer_segments.csv")
    for i, row in customer.iterrows():
        rows.append({
            "section": "Customer Segment",
            "dimension": str(row["customer_segment"]).replace("_", " ").title(),
            "metric": "GMV Share",
            "value": row["merchandise_value_share_pct"],
            "secondary_value": row["customer_share_pct"],
            "sort_order": next_sort + int(i),
        })
    next_sort += len(customer)

    cohort = pd.read_csv(DATA / "analysis_retention_curve.csv")
    for i, row in cohort.iterrows():
        rows.append({
            "section": "Cohort Retention",
            "dimension": f"Month {int(row['month_number'])}",
            "metric": "Weighted Retention %",
            "value": row["weighted_retention_pct"],
            "secondary_value": row["observable_cohorts"],
            "sort_order": next_sort + int(i),
        })
    next_sort += len(cohort)

    delivery = pd.read_csv(DATA / "analysis_delivery_impact.csv")
    for i, row in delivery.iterrows():
        rows.append({
            "section": "Delay Impact",
            "dimension": row["delivery_bucket"],
            "metric": "Average Review",
            "value": row["average_review_score"],
            "secondary_value": row["one_star_review_pct"],
            "sort_order": next_sort + int(i),
        })
    next_sort += len(delivery)

    priority = pd.read_csv(DATA / "analysis_operational_priority.csv").head(15)
    for i, row in priority.iterrows():
        rows.append({
            "section": "Operational Priority",
            "dimension": f"{row['entity_name']} ({row['entity_type']})",
            "metric": "Late Order GMV",
            "value": row["late_order_merchandise_value_brl"],
            "secondary_value": row["late_delivery_pct"],
            "sort_order": next_sort + int(i),
        })

    enriched = pd.concat([base, pd.DataFrame(rows)], ignore_index=True)
    enriched.to_csv(TABLEAU, index=False)

    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    manifest["files"][TABLEAU.name] = {
        "rows": int(len(enriched)),
        "columns": list(enriched.columns),
        "sha256": sha256(TABLEAU),
    }
    MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Tableau deep-analysis extract: {len(enriched)} rows")


if __name__ == "__main__":
    main()


### `refresh_verified_data.py`


In [ ]:
"""Rebuild the pinned Olist warehouse and refresh both BI dashboard data sets.

This is the recruiter-facing reproducibility path for the BI project. It reuses the
SQL project's real download, warehouse and integrity logic, verifies retained
headline evidence, then feeds the existing Power BI/Tableau export contract.
"""
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
PROJECT = Path(__file__).resolve().parent
ECOMMERCE = ROOT / "projects" / "ecommerce_sql_analytics"
ARTIFACTS = PROJECT / ".artifacts"
SOURCE_TABLES = ARTIFACTS / "verified_tables"
DATA_DIR = PROJECT / "data"
DB_PATH = ARTIFACTS / "ecommerce.duckdb"

sys.path.insert(0, str(ECOMMERCE))
from src.config import ProjectConfig  # noqa: E402
from src.data import ensure_dataset  # noqa: E402
from src.validate import assert_integrity, run_integrity_checks  # noqa: E402
from src.warehouse import build_analytics, connect, load_raw_tables  # noqa: E402

from business_analysis import build_business_analysis  # noqa: E402
from prepare_bi_data import REQUIRED_COLUMNS, build_outputs  # noqa: E402


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def verify_retained_headline(con) -> dict[str, object]:
    """Check the rebuild against the retained executed headline evidence."""
    headline = con.execute("SELECT * FROM analytics.headline_kpis").df().iloc[0].to_dict()
    repeat = con.execute("SELECT * FROM analytics.customer_order_frequency").df().iloc[0].to_dict()
    strongest = con.execute(
        "SELECT order_month, merchandise_value_brl FROM analytics.monthly_performance "
        "ORDER BY merchandise_value_brl DESC LIMIT 1"
    ).fetchone()

    observed = {
        "commercial_orders": int(headline["commercial_orders"]),
        "unique_customers": int(headline["unique_customers"]),
        "merchandise_value_brl": round(float(headline["merchandise_value_brl"]), 2),
        "repeat_customer_pct": round(float(repeat["repeat_customer_pct"]), 2),
        "strongest_complete_month": str(strongest[0])[:7],
        "strongest_month_merchandise_value_brl": round(float(strongest[1]), 2),
    }
    retained = json.loads(
        (ECOMMERCE / "results" / "verified_summary.json").read_text(encoding="utf-8")
    )["headline_metrics"]
    expected = {
        "commercial_orders": int(retained["commercial_orders"]),
        "unique_customers": int(retained["unique_customers"]),
        "merchandise_value_brl": round(float(retained["merchandise_value_brl"]), 2),
        "repeat_customer_pct": round(float(retained["repeat_customer_pct"]), 2),
        "strongest_complete_month": str(retained["strongest_complete_month"]),
        "strongest_month_merchandise_value_brl": round(
            float(retained["strongest_month_merchandise_value_brl"]), 2
        ),
    }
    if observed != expected:
        raise AssertionError(
            "Rebuilt warehouse differs from retained executed evidence. "
            f"observed={observed}; expected={expected}"
        )
    return observed


def export_verified_source_tables(con) -> None:
    SOURCE_TABLES.mkdir(parents=True, exist_ok=True)
    for table_name in REQUIRED_COLUMNS:
        frame = con.execute(f"SELECT * FROM analytics.{table_name}").df()
        frame.to_parquet(SOURCE_TABLES / f"{table_name}.parquet", index=False)


def add_operational_exports(con, manifest: dict[str, object]) -> None:
    """Add BI-specific views that make regional and seller risk explorable."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    extras = {
        "state_performance.csv": con.execute(
            """
            SELECT
                customer_state,
                COUNT(*)::BIGINT AS orders,
                COUNT(DISTINCT customer_unique_id)::BIGINT AS unique_customers,
                ROUND(SUM(merchandise_value_brl), 2) AS merchandise_value_brl,
                ROUND(AVG(merchandise_value_brl), 2) AS average_order_value_brl,
                ROUND(100.0 * AVG(CASE WHEN delivered_late THEN 1.0 WHEN delivered_late = FALSE THEN 0.0 END), 2) AS late_delivery_pct,
                ROUND(AVG(review_score), 2) AS average_review_score
            FROM analytics.order_mart
            WHERE commercial_order AND customer_state IS NOT NULL
            GROUP BY customer_state
            ORDER BY merchandise_value_brl DESC
            """
        ).df(),
        "seller_operational_review.csv": con.execute(
            "SELECT * FROM analytics.seller_operational_review"
        ).df(),
    }
    files = manifest.setdefault("files", {})
    for filename, frame in extras.items():
        destination = DATA_DIR / filename
        frame.to_csv(destination, index=False)
        files[filename] = {
            "rows": int(len(frame)),
            "columns": list(frame.columns),
            "sha256": sha256(destination),
        }


def enrich_tableau_story(con, manifest: dict[str, object]) -> None:
    """Extend the common Tableau long extract with regional, payment and risk views."""
    destination = DATA_DIR / "tableau_dashboard_long.csv"
    base = pd.read_csv(destination)
    next_sort = int(base["sort_order"].max()) + 1
    rows: list[dict[str, object]] = []

    states = con.execute(
        """
        SELECT customer_state, merchandise_value_brl, late_delivery_pct
        FROM (
            SELECT
                customer_state,
                ROUND(SUM(merchandise_value_brl), 2) AS merchandise_value_brl,
                ROUND(100.0 * AVG(CASE WHEN delivered_late THEN 1.0 WHEN delivered_late = FALSE THEN 0.0 END), 2) AS late_delivery_pct
            FROM analytics.order_mart
            WHERE commercial_order AND customer_state IS NOT NULL
            GROUP BY customer_state
        )
        ORDER BY merchandise_value_brl DESC
        """
    ).df()
    for rank, row in states.iterrows():
        rows.append({
            "section": "Region",
            "dimension": row["customer_state"],
            "metric": "State GMV",
            "value": row["merchandise_value_brl"],
            "secondary_value": row["late_delivery_pct"],
            "sort_order": next_sort + int(rank),
        })
    next_sort += len(states)

    payments = con.execute("SELECT * FROM analytics.payment_behaviour ORDER BY payment_value_brl DESC").df()
    for rank, row in payments.iterrows():
        rows.append({
            "section": "Payment",
            "dimension": row["payment_type"],
            "metric": "Payment Value",
            "value": row["payment_value_brl"],
            "secondary_value": row["order_penetration_pct"],
            "sort_order": next_sort + int(rank),
        })
    next_sort += len(payments)

    seller_risk = con.execute(
        """
        SELECT operational_status, COUNT(*) AS seller_count, SUM(merchandise_value_brl) AS merchandise_value_brl
        FROM analytics.seller_operational_review
        GROUP BY operational_status
        ORDER BY seller_count DESC
        """
    ).df()
    for rank, row in seller_risk.iterrows():
        rows.append({
            "section": "Seller Risk",
            "dimension": row["operational_status"],
            "metric": "Seller Count",
            "value": row["seller_count"],
            "secondary_value": row["merchandise_value_brl"],
            "sort_order": next_sort + int(rank),
        })

    enriched = pd.concat([base, pd.DataFrame(rows)], ignore_index=True)
    enriched.to_csv(destination, index=False)
    manifest["files"][destination.name] = {
        "rows": int(len(enriched)),
        "columns": list(enriched.columns),
        "sha256": sha256(destination),
    }


def main() -> None:
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    config = ProjectConfig(database_path=DB_PATH, output_dir=ARTIFACTS)
    source_hashes = ensure_dataset(config)

    con = connect(DB_PATH)
    load_raw_tables(con, config.data_dir)
    build_analytics(con, ECOMMERCE / "sql")
    checks = run_integrity_checks(con)
    assert_integrity(checks)
    retained_headline = verify_retained_headline(con)

    export_verified_source_tables(con)
    manifest = build_outputs(SOURCE_TABLES, DATA_DIR)
    add_operational_exports(con, manifest)
    enrich_tableau_story(con, manifest)
    business_summary = build_business_analysis(con, DATA_DIR, manifest)
    manifest["business_analysis"] = business_summary
    manifest["verification"] = {
        "verification_pass": True,
        "dataset_version": config.dataset_version,
        "archive_sha256": config.archive_sha256,
        "source_hashes": source_hashes,
        "headline": retained_headline,
        "integrity_checks": [
            {
                "name": check.name,
                "passed": bool(check.passed),
                "value": check.value,
                "expectation": check.expectation,
            }
            for check in checks
        ],
    }
    (DATA_DIR / "manifest.json").write_text(
        json.dumps(manifest, indent=2, default=str), encoding="utf-8"
    )
    con.close()
    print(json.dumps(manifest["verification"], indent=2, default=str))


if __name__ == "__main__":
    main()


### `render_dashboard_preview.py`


In [ ]:
"""Render the GitHub dashboard preview directly from verified BI analysis outputs."""
from __future__ import annotations

import json
from html import escape
from pathlib import Path

import pandas as pd

PROJECT = Path(__file__).resolve().parent
DATA = PROJECT / "data"
OUTPUT = PROJECT / "dashboard_preview.svg"
WIDTH = 1440
HEIGHT = 1140


def money_short(value: float) -> str:
    if abs(value) >= 1_000_000:
        return f"R${value / 1_000_000:.2f}M"
    if abs(value) >= 1_000:
        return f"R${value / 1_000:.0f}K"
    return f"R${value:,.0f}"


def text(x: float, y: float, value: object, size: int = 18, weight: int = 400, anchor: str = "start", fill: str = "#172033") -> str:
    return (
        f'<text x="{x}" y="{y}" font-family="Inter,Segoe UI,Arial,sans-serif" '
        f'font-size="{size}" font-weight="{weight}" text-anchor="{anchor}" fill="{fill}">{escape(str(value))}</text>'
    )


def rect(x: float, y: float, w: float, h: float, fill: str = "#ffffff", rx: int = 18, stroke: str = "#e7eaf0") -> str:
    return f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="{rx}" fill="{fill}" stroke="{stroke}"/>'


def line_chart(monthly: pd.DataFrame, x: float, y: float, w: float, h: float) -> list[str]:
    values = monthly["merchandise_value_brl"].astype(float).tolist()
    labels = pd.to_datetime(monthly["order_month"]).dt.strftime("%b %y").tolist()
    vmin, vmax = min(values), max(values)
    usable_h = h - 55
    usable_w = w - 20
    points = []
    for i, value in enumerate(values):
        px = x + 10 + usable_w * i / max(1, len(values) - 1)
        py = y + 15 + usable_h * (1 - (value - vmin) / max(1.0, vmax - vmin))
        points.append((px, py))
    svg = [f'<polyline points="{" ".join(f"{px:.1f},{py:.1f}" for px, py in points)}" fill="none" stroke="#2563eb" stroke-width="4" stroke-linecap="round" stroke-linejoin="round"/>']
    for px, py in points:
        svg.append(f'<circle cx="{px:.1f}" cy="{py:.1f}" r="3.5" fill="#2563eb"/>')
    for idx in [0, 5, 10, 15, len(labels) - 1]:
        if 0 <= idx < len(labels):
            px, _ = points[idx]
            svg.append(text(px, y + h - 8, labels[idx], 12, 400, "middle", "#687386"))
    return svg


def retention_chart(curve: pd.DataFrame, x: float, y: float, w: float, h: float) -> list[str]:
    """Plot post-acquisition retention; month zero is omitted so sub-1% values remain legible."""
    view = curve.loc[curve["month_number"].between(1, 12)].copy()
    values = view["weighted_retention_pct"].astype(float).tolist()
    months = view["month_number"].astype(int).tolist()
    if not values:
        return []
    vmax = max(values)
    usable_h = h - 40
    usable_w = w - 20
    points = []
    for i, value in enumerate(values):
        px = x + 10 + usable_w * i / max(1, len(values) - 1)
        py = y + 8 + usable_h * (1 - value / max(vmax, 0.01))
        points.append((px, py))
    svg = [
        f'<line x1="{x + 10}" y1="{y + 8 + usable_h}" x2="{x + w - 10}" y2="{y + 8 + usable_h}" stroke="#e7eaf0" stroke-width="1"/>',
        f'<polyline points="{" ".join(f"{px:.1f},{py:.1f}" for px, py in points)}" fill="none" stroke="#b45309" stroke-width="4" stroke-linecap="round" stroke-linejoin="round"/>',
    ]
    for i, (px, py) in enumerate(points):
        svg.append(f'<circle cx="{px:.1f}" cy="{py:.1f}" r="3.5" fill="#b45309"/>')
        if months[i] in {1, 3, 6, 12}:
            svg.append(text(px, py - 10, f"{values[i]:.2f}%", 12, 700, "middle", "#92400e"))
            svg.append(text(px, y + h - 4, f"M{months[i]}", 11, 500, "middle", "#687386"))
    return svg


def horizontal_bars(frame: pd.DataFrame, label_col: str, value_col: str, x: float, y: float, w: float, row_h: float, formatter, bar_fill: str = "#0f766e") -> list[str]:
    svg: list[str] = []
    max_value = max(float(frame[value_col].max()), 1.0)
    label_w = 120
    for i, row in frame.reset_index(drop=True).iterrows():
        yy = y + i * row_h
        label = str(row[label_col])
        value = float(row[value_col])
        svg.append(text(x, yy + 18, label, 14, 500, "start", "#384152"))
        bx = x + label_w
        bw = (w - label_w - 95) * value / max_value
        svg.append(f'<rect x="{bx}" y="{yy + 5}" width="{max(2, bw):.1f}" height="16" rx="8" fill="{bar_fill}" opacity="0.88"/>')
        svg.append(text(x + w, yy + 18, formatter(value), 13, 600, "end", "#172033"))
    return svg


def main() -> None:
    summary = json.loads((DATA / "analysis_summary.json").read_text(encoding="utf-8"))
    monthly = pd.read_csv(DATA / "analysis_monthly_growth.csv")
    delivery = pd.read_csv(DATA / "analysis_delivery_impact.csv")
    customers = pd.read_csv(DATA / "analysis_customer_segments.csv")
    retention = pd.read_csv(DATA / "analysis_retention_curve.csv")
    priority = pd.read_csv(DATA / "analysis_operational_priority.csv").head(6)

    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{WIDTH}" height="{HEIGHT}" viewBox="0 0 {WIDTH} {HEIGHT}">',
        '<rect width="100%" height="100%" fill="#f6f8fb"/>',
        text(48, 54, "Executive Commerce Intelligence", 30, 700),
        text(48, 84, "Verified Olist marketplace analysis · Power BI + Tableau portfolio", 15, 400, "start", "#687386"),
        text(1392, 54, "GitHub-generated from verified data", 13, 600, "end", "#2563eb"),
    ]

    cards = [
        ("Commercial orders", f"{summary['scope']['commercial_orders']:,}", "#2563eb"),
        ("Unique customers", f"{summary['scope']['unique_customers']:,}", "#0f766e"),
        ("Merchandise value", money_short(summary['scope']['merchandise_value_brl']), "#7c3aed"),
        ("Repeat customers", f"{summary['customers']['repeat_customer_pct']:.2f}%", "#b45309"),
    ]
    card_w = 318
    for i, (label, value, accent) in enumerate(cards):
        x = 48 + i * 338
        parts.append(rect(x, 112, card_w, 116))
        parts.append(f'<rect x="{x}" y="112" width="7" height="116" rx="4" fill="{accent}" stroke="none"/>')
        parts.append(text(x + 24, 150, label, 14, 600, "start", "#687386"))
        parts.append(text(x + 24, 198, value, 30, 700))

    parts.append(rect(48, 252, 850, 300))
    parts.append(text(72, 286, "Commercial growth", 20, 700))
    parts.append(text(72, 310, "Complete-month GMV · Jan 2017 to Aug 2018", 13, 400, "start", "#687386"))
    parts.extend(line_chart(monthly, 72, 330, 802, 195))
    peak = summary["growth"]
    parts.append(text(850, 286, f"Peak {peak['strongest_complete_month']}", 13, 600, "end", "#2563eb"))
    parts.append(text(850, 308, money_short(peak["strongest_month_merchandise_value_brl"]), 17, 700, "end"))

    parts.append(rect(922, 252, 470, 300))
    parts.append(text(946, 286, "Delivery → customer experience", 20, 700))
    parts.append(text(946, 310, "Average review score by delivery delay", 13, 400, "start", "#687386"))
    parts.extend(horizontal_bars(delivery, "delivery_bucket", "average_review_score", 946, 336, 420, 39, lambda v: f"{v:.2f}/5", "#dc2626"))
    parts.append(text(946, 532, f"Late-order share: {summary['delivery']['late_delivery_share_pct']:.2f}% · review gap: {summary['delivery']['review_score_gap']:.2f} points", 13, 600, "start", "#9f1239"))

    parts.append(rect(48, 576, 610, 344))
    parts.append(text(72, 610, "Customer value structure", 20, 700))
    parts.append(text(72, 634, "GMV share by customer segment", 13, 400, "start", "#687386"))
    customer_view = customers[["customer_segment", "merchandise_value_share_pct"]].copy()
    customer_view["customer_segment"] = customer_view["customer_segment"].str.replace("_", " ").str.title()
    parts.extend(horizontal_bars(customer_view, "customer_segment", "merchandise_value_share_pct", 72, 662, 560, 46, lambda v: f"{v:.1f}%", "#7c3aed"))
    parts.append(text(72, 876, "Only 3.03% of customers repeat; high-value one-time buyers dominate value.", 13, 600, "start", "#5b21b6"))

    parts.append(rect(682, 576, 710, 344))
    parts.append(text(706, 610, "Operational priority", 20, 700))
    parts.append(text(706, 634, "Material value currently attached to late orders", 13, 400, "start", "#687386"))
    priority_view = priority.copy()
    priority_view["entity_name"] = priority_view.apply(lambda r: f"{r['entity_name']} ({r['entity_type']})", axis=1)
    parts.extend(horizontal_bars(priority_view, "entity_name", "late_order_merchandise_value_brl", 706, 660, 650, 37, money_short, "#0f766e"))
    parts.append(text(706, 892, "Priority is ranked by late-order GMV — transparent, auditable, and decision-oriented.", 13, 600, "start", "#0f766e"))

    # Retention is separated from the customer-value bars so the very low post-purchase
    # percentages remain legible rather than being visually crushed by the 100% M0 baseline.
    parts.append(rect(48, 944, 1344, 148))
    parts.append(text(72, 976, "Cohort retention after acquisition", 20, 700))
    parts.append(text(72, 1000, "Right-censored eligible cohorts · observed zero-return months retained", 13, 400, "start", "#687386"))
    parts.extend(retention_chart(retention, 500, 962, 850, 105))
    cohort = summary["cohort_retention"]
    parts.append(text(72, 1038, f"M1 {cohort['month_1_weighted_retention_pct']:.2f}%", 19, 700, "start", "#92400e"))
    parts.append(text(190, 1038, f"M3 {cohort['month_3_weighted_retention_pct']:.2f}%", 19, 700, "start", "#92400e"))
    parts.append(text(308, 1038, f"M6 {cohort['month_6_weighted_retention_pct']:.2f}%", 19, 700, "start", "#92400e"))
    parts.append(text(426, 1038, f"M12 {cohort['month_12_weighted_retention_pct']:.2f}%", 19, 700, "start", "#92400e"))
    parts.append(text(72, 1068, "Retention remains below 0.5% after month 1 — a clear customer-lifecycle investigation area.", 12, 600, "start", "#92400e"))

    parts.append(text(48, 1122, "Source: Olist Brazilian e-commerce public dataset · pinned dataset version + hash checks · generated by GitHub Actions", 12, 400, "start", "#7b8495"))
    parts.append("</svg>")
    OUTPUT.write_text("\n".join(parts), encoding="utf-8")
    print(f"Wrote {OUTPUT}")


if __name__ == "__main__":
    main()


## Run the real project

The cell below executes the canonical project entry point rather than a rewritten toy version. Keep `RUN_PIPELINE = False` when you only want to inspect the notebook; change it to `True` to reproduce the project.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print(f'Reproduce with: cd {PROJECT} && python run.py')


In [ ]:
evidence = []
for folder in (PROJECT / 'results', ROOT / 'verified' / PROJECT_SLUG):
    if folder.exists():
        evidence.extend(sorted(folder.glob('*.json')))
for path in evidence[:5]:
    print('\n---', path.relative_to(ROOT), '---')
    print(path.read_text(encoding='utf-8')[:12000])


## Interview discussion

Be ready to explain the business problem, dataset provenance, cleaning/preprocessing decisions, leakage controls, modelling or analytical choices, evaluation design, limitations, testing strategy and what you would change in production. The key signal is that the notebook, modular source, tests and retained evidence all tell the same story.


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `tests/test_project_contract.py`


In [ ]:
from __future__ import annotations

import json
import unittest
import xml.etree.ElementTree as ET
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]


class ExecutiveCommerceBIContractTests(unittest.TestCase):
    def test_required_assets_exist(self) -> None:
        required = [
            "README.md",
            "KPI_DICTIONARY.md",
            "DASHBOARD_STORY.md",
            "dashboard_preview.svg",
            "business_analysis.py",
            "cohort_analysis.py",
            "enrich_tableau_analysis.py",
            "render_dashboard_preview.py",
            "refresh_verified_data.py",
            "power_bi/ExecutiveCommerce.pbip",
            "power_bi/ExecutiveCommerce.SemanticModel/definition.pbism",
            "power_bi/ExecutiveCommerce.SemanticModel/definition/model.tmdl",
            "power_bi/ExecutiveCommerce.Report/definition.pbir",
            "tableau/ExecutiveCommerce.twb",
        ]
        for relative in required:
            self.assertTrue((ROOT / relative).exists(), relative)

    def test_power_bi_bindings_and_three_page_story(self) -> None:
        pbip = json.loads((ROOT / "power_bi/ExecutiveCommerce.pbip").read_text())
        self.assertEqual(pbip["artifacts"][0]["report"]["path"], "ExecutiveCommerce.Report")
        pbir = json.loads((ROOT / "power_bi/ExecutiveCommerce.Report/definition.pbir").read_text())
        self.assertEqual(pbir["datasetReference"]["byPath"]["path"], "../ExecutiveCommerce.SemanticModel")
        pages = json.loads((ROOT / "power_bi/ExecutiveCommerce.Report/definition/pages/pages.json").read_text())
        self.assertIn(pages["activePageName"], pages["pageOrder"])
        self.assertEqual(3, len(pages["pageOrder"]))
        page_names = set()
        for page_id in pages["pageOrder"]:
            page = json.loads(
                (ROOT / f"power_bi/ExecutiveCommerce.Report/definition/pages/{page_id}/page.json").read_text()
            )
            page_names.add(page["displayName"])
        self.assertEqual(
            {"Executive Overview", "Market & Operations", "Customer & Service Drivers"},
            page_names,
        )

        drivers = ROOT / "power_bi/ExecutiveCommerce.Report/definition/pages/9c2f4e6a8b1d3f5c7e0a/visuals"
        driver_source = "\n".join(path.read_text() for path in drivers.rglob("visual.json"))
        for entity in ["CustomerSegments", "CohortRetention", "DeliveryImpact", "OperationalPriority"]:
            self.assertIn(entity, driver_source)

    def test_power_bi_semantic_model_has_business_depth(self) -> None:
        model = (ROOT / "power_bi/ExecutiveCommerce.SemanticModel/definition/model.tmdl").read_text()
        for table in [
            "ExecutiveKPIs",
            "MonthlyPerformance",
            "CategoryPerformance",
            "DeliveryQuality",
            "StatePerformance",
            "PaymentBehaviour",
            "SellerOperations",
            "CustomerSegments",
            "CohortRetention",
            "DeliveryImpact",
            "OperationalPriority",
        ]:
            self.assertIn(f"ref table {table}", model)

        delivery = (ROOT / "power_bi/ExecutiveCommerce.SemanticModel/definition/tables/DeliveryQuality.tmdl").read_text()
        self.assertIn("measure 'Late Delivery Rate'", delivery)
        self.assertIn("measure 'Delivery Review Gap'", delivery)
        sellers = (ROOT / "power_bi/ExecutiveCommerce.SemanticModel/definition/tables/SellerOperations.tmdl").read_text()
        self.assertIn("measure 'Priority Sellers'", sellers)
        segments = (ROOT / "power_bi/ExecutiveCommerce.SemanticModel/definition/tables/CustomerSegments.tmdl").read_text()
        self.assertIn("measure 'Segment GMV Share'", segments)
        cohort = (ROOT / "power_bi/ExecutiveCommerce.SemanticModel/definition/tables/CohortRetention.tmdl").read_text()
        self.assertIn("measure 'Weighted Cohort Retention'", cohort)
        impact = (ROOT / "power_bi/ExecutiveCommerce.SemanticModel/definition/tables/DeliveryImpact.tmdl").read_text()
        self.assertIn("measure 'One Star Review Rate'", impact)
        priority = (ROOT / "power_bi/ExecutiveCommerce.SemanticModel/definition/tables/OperationalPriority.tmdl").read_text()
        self.assertIn("measure 'Late Order GMV'", priority)

    def test_all_pbir_json_is_valid(self) -> None:
        report = ROOT / "power_bi/ExecutiveCommerce.Report"
        for path in report.rglob("*.json"):
            with self.subTest(path=path):
                json.loads(path.read_text(encoding="utf-8"))

    def test_tableau_xml_and_dashboards(self) -> None:
        tree = ET.parse(ROOT / "tableau/ExecutiveCommerce.twb")
        dashboard_names = {item.attrib.get("name") for item in tree.findall("./dashboards/dashboard")}
        self.assertEqual(
            {"Executive Commerce Dashboard", "Marketplace Explorer", "Customer & Service Drivers"},
            dashboard_names,
        )
        worksheet_names = {item.attrib.get("name") for item in tree.findall("./worksheets/worksheet")}
        self.assertEqual(
            {
                "Executive Pulse",
                "Monthly Trend",
                "Category Leaders",
                "Delivery Experience",
                "Regional Performance",
                "Payment Mix",
                "Seller Risk",
                "Customer Value",
                "Cohort Retention",
                "Delay Severity",
                "Operational Priority",
            },
            worksheet_names,
        )
        for worksheet in tree.findall("./worksheets/worksheet"):
            self.assertIsNotNone(worksheet.find("./table/view"))
            self.assertIsNotNone(worksheet.find("./table/panes/pane/mark"))
            self.assertIsNotNone(worksheet.find("./table/rows"))
            self.assertIsNotNone(worksheet.find("./table/cols"))

    def test_preview_uses_verified_analysis_values(self) -> None:
        preview = (ROOT / "dashboard_preview.svg").read_text()
        for value in [
            "98,199",
            "94,983",
            "R$13.49M",
            "3.03%",
            "8.11%",
            "1.74",
            "R$342K",
            "Cohort retention after acquisition",
            "M1 0.48%",
            "M3 0.26%",
            "M6 0.23%",
            "M12 0.18%",
        ]:
            self.assertIn(value, preview)

    def test_generated_data_when_present(self) -> None:
        manifest_path = ROOT / "data/manifest.json"
        if not manifest_path.exists():
            self.skipTest("Generated dashboard extracts are created by the BI refresh workflow")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        self.assertTrue(manifest["verification"]["verification_pass"])
        self.assertEqual(98199, manifest["verification"]["headline"]["commercial_orders"])
        for filename in [
            "executive_kpis.csv",
            "monthly_performance.csv",
            "category_performance.csv",
            "delivery_review_summary.csv",
            "payment_behaviour.csv",
            "state_performance.csv",
            "seller_operational_review.csv",
            "tableau_dashboard_long.csv",
            "analysis_summary.json",
            "analysis_monthly_growth.csv",
            "analysis_customer_segments.csv",
            "analysis_cohort_retention.csv",
            "analysis_retention_curve.csv",
            "analysis_delivery_impact.csv",
            "analysis_state_performance.csv",
            "analysis_category_performance.csv",
            "analysis_seller_risk.csv",
            "analysis_payment_mix.csv",
            "analysis_operational_priority.csv",
        ]:
            self.assertIn(filename, manifest["files"])


if __name__ == "__main__":
    unittest.main()


# Portfolio depth check

**Meaningful code lines visible in this notebook:** 1,337. For a major recruiter-facing application the working target is roughly **1,000 meaningful lines**, with a practical guide of about 600–1,400 depending on the problem. This notebook is within the major-project guide.

Line count is not a quality metric by itself. Add code only when it improves the real project: data acquisition, validation, cleaning, EDA, visualisation, feature engineering, baselines, model comparison, tuning, leakage control, error analysis, explainability, uncertainty, inference, tests, monitoring, deployment or decision logic.
